# Synthetic Title Prompt Lab

Use this notebook for fast prompt/model experiments with individual LLM calls.

- Edit `system_prompt`, `user_prompt_template`, and `model_name`.
- Run one call and inspect raw + parsed outputs.
- Optionally compare several models on the same prompt.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "synthetic_data").exists():
    ROOT = ROOT.parent
if not (ROOT / "synthetic_data").exists():
    raise RuntimeError("Run this notebook from the repo root or from synthetic_data/.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

ROOT

In [ ]:
import asyncio
import json
import pandas as pd

from synthetic_data.clients import build_client, resolve_model, supported_model_names
from synthetic_data.generate import (
    build_thinking_config,
    load_params,
    load_seed_titles,
    normalize_openai_reasoning_effort,
    parse_response,
)
from synthetic_data.models import TitleVariants

In [ ]:
params = load_params("params.synthetic_data")
seed_df = load_seed_titles(Path("synthetic_data/data/seed_titles.csv"))

# ----- Experiment config (edit these) -----
model_name = params["model"]
seed_title = seed_df["seed_title"].iloc[0]
num_examples = params["num_examples_per_title"]
temperature = params["temperature"]
reasoning_effort = params["reasoning_effort"]
api_key_env = params["api_key_env"]
base_url_env = params["base_url_env"]

system_prompt = (
    "You generate realistic, varied job-posting titles. Titles should resemble listings "
    "seen on job boards: occasional locations, teams, seniority, bonuses, or abbreviations. "
    "Avoid repeating a single pattern."
)

user_prompt_template = (
    "Seed title: {seed_title}\n"
    "Return exactly {num_examples} distinct 'in-the-wild' job titles as JSON using schema {schema}.\n"
    "Do not include any prose outside the JSON."
)

print("Selected model:", model_name)
print("Available models:", ", ".join(sorted(supported_model_names())))
print("Seed title:", seed_title)
print("Temperature:", temperature)
print("Reasoning effort:", reasoning_effort)
print("API key env:", api_key_env)
print("Base URL env:", base_url_env)

In [ ]:
async def generate_once(
    model_name: str,
    seed_title: str,
    num_examples: int,
    system_prompt: str,
    user_prompt_template: str,
    api_key_env: str | None,
    base_url_env: str | None,
    temperature: float | None,
    reasoning_effort: str,
):
    model_info = resolve_model(model_name)
    client = build_client(
        model_info,
        api_key_env=api_key_env,
        base_url_env=base_url_env,
        async_mode=True,
    )

    schema = json.dumps(TitleVariants.model_json_schema())
    user_prompt = user_prompt_template.format(
        seed_title=seed_title,
        num_examples=num_examples,
        schema=schema,
    )

    if model_info.provider == "openai":
        normalized_effort = normalize_openai_reasoning_effort(model_info.model, reasoning_effort)
        kwargs = {
            "model": model_info.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "response_format": {"type": "json_object"},
        }
        if normalized_effort is not None:
            kwargs["reasoning_effort"] = normalized_effort
        if temperature is not None:
            kwargs["temperature"] = temperature
        completion = await client.chat.completions.create(**kwargs)
        raw_payload = completion.choices[0].message.content or ""
    else:
        config = {"response_mime_type": "application/json"} | build_thinking_config(
            model_info.model,
            reasoning_effort,
        )
        if temperature is not None:
            config["temperature"] = temperature
        response = await asyncio.to_thread(
            client.models.generate_content,
            model=model_info.model,
            contents="\n\n".join([system_prompt, user_prompt]),
            config=config,
        )
        raw_payload = response.text or ""

    parsed = parse_response(seed_title, raw_payload)
    return model_info, user_prompt, raw_payload, parsed

In [ ]:
# Single-call experiment
model_info, rendered_user_prompt, raw_payload, parsed = await generate_once(
    model_name=model_name,
    seed_title=seed_title,
    num_examples=num_examples,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    api_key_env=api_key_env,
    base_url_env=base_url_env,
    temperature=temperature,
    reasoning_effort=reasoning_effort,
)

print(f"Provider: {model_info.provider}")
print(f"Canonical model: {model_info.model}")
print(f"Returned titles: {len(parsed.in_the_wild_titles)}")

In [ ]:
print("=== Rendered User Prompt ===")
print(rendered_user_prompt)
print("\n=== Raw Response Payload ===")
print(raw_payload)
print("\n=== Parsed Titles ===")
pd.DataFrame({"generated_title": parsed.in_the_wild_titles})

## Optional: Compare Multiple Models

Uncomment model names below and run this cell to do one call per model with the same prompt.

In [ ]:
compare_models = [
    "gpt-5-mini",
    "gpt-5.2",
    # "gemini-3-flash-preview",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
]

rows = []
for name in compare_models:
    try:
        info, _, raw, parsed_obj = await generate_once(
            model_name=name,
            seed_title=seed_title,
            num_examples=num_examples,
            system_prompt=system_prompt,
            user_prompt_template=user_prompt_template,
            api_key_env=api_key_env,
            base_url_env=base_url_env,
            temperature=temperature,
            reasoning_effort=reasoning_effort,
        )
        rows.append(
            {
                "model": info.model,
                "provider": info.provider,
                "n_titles": len(parsed_obj.in_the_wild_titles),
                "sample_title": parsed_obj.in_the_wild_titles[0] if parsed_obj.in_the_wild_titles else "",
                "status": "ok",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": name,
                "provider": "",
                "n_titles": 0,
                "sample_title": "",
                "status": f"error: {exc}",
            }
        )

pd.DataFrame(rows)